In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/bengali-nlp-safety-challenge-2026/sample_submission.csv
/kaggle/input/competitions/bengali-nlp-safety-challenge-2026/validation.csv
/kaggle/input/competitions/bengali-nlp-safety-challenge-2026/train.csv
/kaggle/input/competitions/bengali-nlp-safety-challenge-2026/test.csv


In [5]:
import os, re, gc, random, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import scipy.sparse as sp

DATA_DIR = "/kaggle/input/competitions/bengali-nlp-safety-challenge-2026"  # tomar path
L = ['is_toxic','implicit_toxicity','sarcasm','contains_profanity','threatening','identity_attack','sexual_content']
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)

def clean(t):
    t = str(t); t = re.sub(r"\s+", " ", t); return t.strip()

tr = pd.read_csv(f"{DATA_DIR}/train.csv")
va = pd.read_csv(f"{DATA_DIR}/validation.csv")
te = pd.read_csv(f"{DATA_DIR}/test.csv")
al = pd.concat([tr, va], ignore_index=True)          # sob train+val data ekshathe, CV e babohar hobe
for d in [al, te]: d["text"] = d["text"].fillna("").apply(clean)

print(DEVICE, al.shape, te.shape)

cuda (3797, 9) (1000, 2)


In [6]:
class ToxDS(Dataset):
    def __init__(s, df, tok, max_len, lab=True):
        s.enc = tok(df.text.tolist(), truncation=True, max_length=max_len)
        s.y = df[L].values.astype("float32") if lab else None
    def __len__(s): return len(s.enc["input_ids"])
    def __getitem__(s, i):
        d = {k: v[i] for k, v in s.enc.items()}
        if s.y is not None: d["labels"] = s.y[i]
        return d

def make_collate(tok):
    def collate(b):
        m = max(len(x["input_ids"]) for x in b)
        out = {"input_ids": torch.tensor([x["input_ids"] + [tok.pad_token_id]*(m-len(x["input_ids"])) for x in b]),
               "attention_mask": torch.tensor([[1]*len(x["input_ids"]) + [0]*(m-len(x["input_ids"])) for x in b])}
        if "labels" in b[0]: out["labels"] = torch.tensor(np.stack([x["labels"] for x in b]))
        return out
    return collate

In [7]:
class ToxModel(nn.Module):
    def __init__(s, model_name, dropout=0.2):
        super().__init__()
        s.enc = AutoModel.from_pretrained(model_name)
        h = s.enc.config.hidden_size
        s.drops = nn.ModuleList([nn.Dropout(p) for p in [0.1, 0.2, 0.3, 0.4, 0.5]])
        s.head = nn.Linear(h * 2, 7) # h*2 because of Mean + Max Pooling

    def forward(s, input_ids, attention_mask):
        out = s.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).float()
        
        # Mean Pooling
        mean_pooled = (out * m).sum(1) / m.sum(1).clamp(min=1e-6)
        
        # Max Pooling
        masked_out = out.masked_fill(m == 0, -1e9)
        max_pooled = torch.max(masked_out, dim=1)[0]
        
        # Concatenate Mean + Max Pooling
        pooled = torch.cat([mean_pooled, max_pooled], dim=1)
        
        logits = torch.stack([s.head(d(pooled)) for d in s.drops], dim=0).mean(0)
        return logits

In [8]:
def f1_total(Y, P, thr=0.5):
    f = [f1_score(Y[:,j], P[:,j] >= thr, average="macro" if j==0 else "binary", zero_division=0) for j in range(7)]
    return 0.5*f[0] + 0.5*np.mean(f[1:]), f

@torch.no_grad()
def predict(model, ds, collate, bs=64):
    model.eval(); P = []
    for b in DataLoader(ds, batch_size=bs, collate_fn=collate):
        b = {k: v.to(DEVICE) for k, v in b.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type=="cuda"):
            p = torch.sigmoid(model(b["input_ids"], b["attention_mask"]))
        P.append(p.float().cpu().numpy())
    return np.concatenate(P)

def train_fold(model_name, tr_df, va_df, epochs=7, bs=16, lr=2e-5, max_len=160):
    tok = AutoTokenizer.from_pretrained(model_name)
    collate = make_collate(tok)
    dtr, dva = ToxDS(tr_df, tok, max_len), ToxDS(va_df, tok, max_len)
    model = ToxModel(model_name).to(DEVICE)
    pos = tr_df[L].values.mean(0)
    pos_w = torch.tensor(np.clip((1 - pos) / np.clip(pos, 1e-6, None), 1.0, 10.0), dtype=torch.float32).to(DEVICE)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loader = DataLoader(dtr, batch_size=bs, shuffle=True, collate_fn=collate)
    total_steps = len(loader) * epochs
    sched = get_cosine_schedule_with_warmup(opt, int(0.1*total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type=="cuda")
    best_score, best_probs = -1, None
    for ep in range(epochs):
        model.train()
        for b in loader:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type=="cuda"):
                loss = lossf(model(b["input_ids"], b["attention_mask"]).float(), b["labels"])
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        pv = predict(model, dva, collate)
        s, _ = f1_total(va_df[L].values, pv)
        print(f"  epoch {ep+1}/{epochs} score@0.5={s:.4f}", flush=True)
        if s > best_score: best_score, best_probs, best_model = s, pv, {k: v.cpu() for k, v in model.state_dict().items()}
    model.load_state_dict(best_model)
    return model, tok, collate, best_probs, best_score

In [9]:
def run_cv(model_name, tag, epochs=7, bs=16, lr=2e-5, max_len=160, folds=5):
    Y = al[L].values
    oof = np.zeros((len(al), 7), dtype="float32")
    test_accum = np.zeros((len(te), 7), dtype="float32")
    kf = KFold(folds, shuffle=True, random_state=SEED)
    for fold, (a, b) in enumerate(kf.split(al)):
        print(f"--- {tag} fold {fold+1}/{folds} ---")
        model, tok, collate, val_probs, val_score = train_fold(model_name, al.iloc[a], al.iloc[b], epochs, bs, lr, max_len)
        oof[b] = val_probs
        dte = ToxDS(te, tok, max_len, lab=False)
        test_accum += predict(model, dte, collate) / folds
        np.save(f"/kaggle/working/oof_{tag}.npy", oof)
        np.save(f"/kaggle/working/test_{tag}.npy", test_accum * folds / (fold+1))
        del model; gc.collect(); torch.cuda.empty_cache()
    score, per_label = f1_total(Y, oof)
    print(f"{tag} OOF score@0.5 = {score:.4f}", per_label)
    return oof, test_accum

In [10]:
oof_bb, test_bb = run_cv("csebuetnlp/banglabert", "banglabert", epochs=7, bs=16, lr=2e-5)

--- banglabert fold 1/5 ---


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
electra.embeddings.position_ids                   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 1/7 score@0.5=0.3243
  epoch 2/7 score@0.5=0.5069
  epoch 3/7 score@0.5=0.5446
  epoch 4/7 score@0.5=0.5481
  epoch 5/7 score@0.5=0.5595
  epoch 6/7 score@0.5=0.5597
  epoch 7/7 score@0.5=0.5587
--- banglabert fold 5/5 ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
electra.embeddings.position_ids                   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 1/7 score@0.5=0.3240
  epoch 2/7 score@0.5=0.4922
  epoch 3/7 score@0.5=0.5418
  epoch 4/7 score@0.5=0.5635
  epoch 5/7 score@0.5=0.5604
  epoch 6/7 score@0.5=0.5498
  epoch 7/7 score@0.5=0.5521
banglabert OOF score@0.5 = 0.5719 [0.733388341886033, 0.22714681440443213, 0.31060272767267927, 0.5518453427065027, 0.3939393939393939, 0.6126076089975007, 0.3669064748201439]


In [11]:
oof_xlmr, test_xlmr = run_cv("xlm-roberta-base", "xlmr", epochs=7, bs=16, lr=2e-5)

--- xlmr fold 1/5 ---


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 1/7 score@0.5=0.3009
  epoch 2/7 score@0.5=0.3538
  epoch 3/7 score@0.5=0.4577
  epoch 4/7 score@0.5=0.4218
  epoch 5/7 score@0.5=0.4732
  epoch 6/7 score@0.5=0.4824
  epoch 7/7 score@0.5=0.4940
--- xlmr fold 5/5 ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 1/7 score@0.5=0.2906
  epoch 2/7 score@0.5=0.3947
  epoch 3/7 score@0.5=0.4769
  epoch 4/7 score@0.5=0.4982
  epoch 5/7 score@0.5=0.5291
  epoch 6/7 score@0.5=0.5243
  epoch 7/7 score@0.5=0.5278
xlmr OOF score@0.5 = 0.5269 [0.6841952351076225, 0.18791946308724833, 0.3114273754709083, 0.5324909747292419, 0.20588235294117646, 0.6023230088495575, 0.3781818181818182]


In [12]:
oof_blish, test_blish = run_cv("csebuetnlp/banglishbert", "banglishbert", epochs=7, bs=16, lr=2e-5)

--- banglishbert fold 1/5 ---


config.json:   0%|          | 0.00/874 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

ElectraModel LOAD REPORT from: csebuetnlp/banglishbert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
electra.embeddings.position_ids                   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 1/7 score@0.5=0.2815
  epoch 2/7 score@0.5=0.4570
  epoch 3/7 score@0.5=0.4916
  epoch 4/7 score@0.5=0.5373
  epoch 5/7 score@0.5=0.5563
  epoch 6/7 score@0.5=0.5571
  epoch 7/7 score@0.5=0.5572
--- banglishbert fold 5/5 ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraModel LOAD REPORT from: csebuetnlp/banglishbert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
electra.embeddings.position_ids                   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 1/7 score@0.5=0.3194
  epoch 2/7 score@0.5=0.4584
  epoch 3/7 score@0.5=0.4942
  epoch 4/7 score@0.5=0.5245
  epoch 5/7 score@0.5=0.5411
  epoch 6/7 score@0.5=0.5399
  epoch 7/7 score@0.5=0.5358
banglishbert OOF score@0.5 = 0.5507 [0.7268578851563103, 0.23255813953488372, 0.31199569661108123, 0.5190989226248776, 0.2555066079295154, 0.605248464544947, 0.3233256351039261]


In [18]:
def tfidf_cv():
    Y = al[L].values
    oof = np.zeros((len(al), 7)); test_accum = np.zeros((len(te), 7))
    kf = KFold(5, shuffle=True, random_state=SEED)
    for a, b in kf.split(al):
        w = TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2, sublinear_tf=True, token_pattern=r"(?u)\b\w+\b")
        c = TfidfVectorizer(analyzer="char_wb", ngram_range=(1,5), min_df=2, sublinear_tf=True, max_features=300000)
        Xa = sp.hstack([w.fit_transform(al.text.iloc[a]), c.fit_transform(al.text.iloc[a])]).tocsr()
        Xb = sp.hstack([w.transform(al.text.iloc[b]), c.transform(al.text.iloc[b])]).tocsr()
        Xt = sp.hstack([w.transform(te.text), c.transform(te.text)]).tocsr()
        for j, lab in enumerate(L):
            m = LogisticRegression(C=1, class_weight="balanced", max_iter=2000).fit(Xa, Y[a, j])
            oof[b, j] = m.predict_proba(Xb)[:, 1]
            test_accum[:, j] += m.predict_proba(Xt)[:, 1] / 5
    return oof, test_accum

oof_tf, test_tf = tfidf_cv()
np.save("/kaggle/working/oof_tfidf.npy", oof_tf); np.save("/kaggle/working/test_tfidf.npy", test_tf)
print("tfidf oof score", f1_total(al[L].values, oof_tf)[0])

tfidf oof score 0.5446814852969261


In [17]:
import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

from scipy.optimize import minimize



Y = al[L].values

oofs  = {"banglabert": oof_bb, "xlmr": oof_xlmr, "banglishbert": oof_blish, "tfidf": oof_tf}

tests = {"banglabert": test_bb, "xlmr": test_xlmr, "banglishbert": test_blish, "tfidf": test_tf}

names = list(oofs.keys())



num_models = len(names)

num_labels = len(L)



best_weights = np.zeros((num_labels, num_models))

best_thresholds = np.zeros(num_labels)



oof_blend = np.zeros_like(Y, dtype=np.float32)

test_blend = np.zeros((len(te), num_labels), dtype=np.float32)



print("--- Optimizing Per-Label Weights and Thresholds ---")



for j in range(num_labels):

    y_j = Y[:, j]

    avg_type = "macro" if j == 0 else "binary"

    

    preds_j = np.column_stack([oofs[n][:, j] for n in names])

    test_preds_j = np.column_stack([tests[n][:, j] for n in names])

    

    def objective(params):

        w = params[:num_models]

        if w.sum() > 0:

            w = w / w.sum()

        else:

            w = np.ones(num_models) / num_models

        

        t = params[num_models]

        p_blend = preds_j @ w

        pred_bin = (p_blend >= t).astype(int)

        

        # Maximize F1 score by returning negative score

        return -f1_score(y_j, pred_bin, average=avg_type, zero_division=0)



    # Initial search params (equal weights, threshold=0.40)

    init_params = np.array([1.0 / num_models] * num_models + [0.40])

    bounds = [(0, 1)] * num_models + [(0.05, 0.85)]

    

    res = minimize(objective, init_params, method='Powell', bounds=bounds)

    

    w_opt = res.x[:num_models]

    w_opt = w_opt / w_opt.sum() if w_opt.sum() > 0 else np.ones(num_models) / num_models

    t_opt = np.clip(res.x[num_models], 0.05, 0.85)

    

    best_weights[j, :] = w_opt

    best_thresholds[j] = t_opt

    

    oof_blend[:, j] = preds_j @ w_opt

    test_blend[:, j] = test_preds_j @ w_opt



# Evaluate final tuned performance

pred_oof_bin = (oof_blend >= best_thresholds).astype(int)

per_label_f1 = [

    f1_score(Y[:, j], pred_oof_bin[:, j], average="macro" if j == 0 else "binary", zero_division=0)

    for j in range(num_labels)

]

final_cv_score = 0.5 * per_label_f1[0] + 0.5 * np.mean(per_label_f1[1:])



print(f"\nTarget Tuned CV Score: {final_cv_score:.4f}")

print("Per-label F1 scores:", [round(s, 4) for s in per_label_f1])

print("\nOptimized Thresholds per Label:")

print(dict(zip(L, np.round(best_thresholds, 3))))



# Generate submission file

pred_test = (test_blend >= best_thresholds).astype(int)

sub = pd.DataFrame(pred_test, columns=L)

sub.insert(0, "ID", te.ID.values)

sub.to_csv("/kaggle/working/submission.csv", index=False)

print("\nSubmission created successfully!")

--- Optimizing Per-Label Weights and Thresholds ---

Target Tuned CV Score: 0.5946
Per-label F1 scores: [0.7526, 0.2625, 0.3401, 0.5806, 0.3819, 0.6489, 0.4053]

Optimized Thresholds per Label:
{'is_toxic': np.float64(0.529), 'implicit_toxicity': np.float64(0.482), 'sarcasm': np.float64(0.347), 'contains_profanity': np.float64(0.451), 'threatening': np.float64(0.387), 'identity_attack': np.float64(0.396), 'sexual_content': np.float64(0.589)}

Submission created successfully!


In [15]:
Pv_safe = 0.5*oofs['banglabert'] + 0.5*oofs['tfidf']
Pt_safe = 0.5*tests['banglabert'] + 0.5*tests['tfidf']

ts = np.arange(0.05, 0.95, 0.01)
T_safe = []
for j in range(7):
    f = np.array([f1_score(Y[:,j], Pv_safe[:,j] >= t, average="macro" if j==0 else "binary", zero_division=0) for t in ts])
    fs = np.convolve(f, np.ones(7)/7, mode="same")
    T_safe.append(ts[int(np.argmax(fs))])
pred_oof = (Pv_safe >= np.array(T_safe)).astype(int)
f = [f1_score(Y[:,j], pred_oof[:,j], average="macro" if j==0 else "binary", zero_division=0) for j in range(7)]
print("safe 50-50 blend score", 0.5*f[0] + 0.5*np.mean(f[1:]), f)

safe 50-50 blend score 0.5894917922173961 [0.7483035243195637, 0.24806201550387597, 0.3343239227340267, 0.5771543086172345, 0.3790322580645161, 0.6355699675729608, 0.40993788819875776]
